In [ ]:
from Base.best_response import BR
from Base.market_clearing import MarketClearing
import pandas as pd
import numpy as np

def update_prof_df(prod_df:pd.DataFrame, pg:pd.Series):
    df = prod_df.set_index('producers')
    prod_df['Pmax'] = pd.concat([df['ramp_constraints']+pg, df['capacities']], axis=1).min(axis=1).to_list()
    prod_df['Pmin'] = pd.concat([-df['ramp_constraints']+pg, pg*0], axis=1).max(axis=1).to_list()
    return prod_df

prod_df = pd.DataFrame({
    'producers': ['P1', 'P2', 'P3', 'P4'],
    'capacities': [40, 90, 50, 60], 
    'Pmax': [40, 90, 50, 60],       
    'Pmin': [0, 0, 0, 0],      
    'marginal_costs': [10, 30, 35, 55],
    'ramp_constraints': [40, 90, 10, 60] #to change
})

marginal_costs = prod_df['marginal_costs'].to_list()
demands = [155, 155, 155]
print(demands)

results_prod = pd.DataFrame()
results_prod.index = prod_df['producers'] #type: ignore
PoAs = []

for (hour, demand) in enumerate(demands):
    br = BR(bids_init=marginal_costs, marginal_costs=marginal_costs, demand=demand, prod_df=prod_df)
    br.run_BR(nb_iter=200, tau_alphas=1)
    results, convergence = br.get_results()
    prod_df = update_prof_df(prod_df=prod_df, pg=results.set_index('producer')['production'])

    # Store prod results
    results_prod[hour] = results.set_index('producer')['production']

    # Calculation Inefficiency
    q_eq = np.array(br.get_equilibrium_dispatch(), dtype=float)
    theta_arr = prod_df['marginal_costs'].to_numpy()
    SC_eq = float(np.dot(theta_arr, q_eq))
    # Verify that the equilibrium dispatch meets the demand
    mc = MarketClearing(bids = prod_df['marginal_costs'].to_list(),
                        marginal_costs=prod_df['marginal_costs'].to_list(), 
                        demand = demand,
                        prod_df = prod_df)
    q_opt = np.array(mc.get_dispatch(), dtype=float)
    SC_opt = float(np.dot(theta_arr, q_opt))    
    # PoA
    PoA = SC_eq / SC_opt
    PoAs.append(PoA)

print(demands)
print(results_prod)
print(prod_df)
print(PoAs)

[155, 155, 155]
Strategic producer P1
Set parameter QCPDual to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Linux Mint 22.2")

CPU model: AMD Ryzen 7 8840HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
QCPDual  1

Optimize a model with 9 rows, 4 columns and 12 nonzeros
Model fingerprint: 0x5845b6b1
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 6e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+01, 2e+02]
Presolve removed 8 rows and 1 columns
Presolve time: 0.00s
Presolved: 1 rows, 3 columns, 3 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.8500000e+03   3.125000e+00   0.000000e+00      0s
       1    3.9750000e+03   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.975000000e+03
Pg : Si